In [16]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
INPUT_JSON = "explanation_output.json"
OUTPUT_JSON = "explanation_output_polished.json"
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [3]:
if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

Using device: mps


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [ ]:
def build_prompt(data):
    query = data["query"]
    overview = data["overview"]
    why_best = data["why_best_overall"]

    prompt = f"""
You are an AI copywriter for an e-commerce site. 
Below is a structured, factual summary of our top product recommendations. 
Rewrite this text to sound natural, engaging, and human. 
CRITICAL: Do not add any new products, features, ratings, or prices that are not present in the original text. 
Maintain the exact factual meaning.

Your job is to improve clarity, flow, and readability without changing the meaning.

Rules:
- Keep the same meaning.
- Do not add new facts.
- Do not mention AI models, semantic scores, ALS, or internal ranking methods.
- Keep the tone natural and helpful.
- Make the overview feel like quick buying guidance.
- Make the "why best overall" explanation feel confident and concise.
- Return valid JSON only.

Input:
Query: {query}

Draft overview:
{overview}

Draft why best overall:
{why_best}

Return valid JSON only:
{{
  "overview": "...",
  "why_best_overall": "..."
}}
"""
    return prompt.strip()

In [13]:
def generate_text(prompt, max_new_tokens=220):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(output[0], skip_special_tokens=True)
    return text


In [14]:
def extract_json(text):
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        return text[start:end+1]
    raise ValueError("No JSON object found in model output")

In [15]:
def main():
    with open(INPUT_JSON, "r") as f:
        data = json.load(f)

    prompt = build_prompt(data)
    raw_output = generate_text(prompt)

    try:
        json_text = extract_json(raw_output)
        polished = json.loads(json_text)
    except Exception:
        polished = {
            "overview": data["overview"],
            "why_best_overall": data["why_best_overall"]
        }

    final_output = {
        **data,
        "overview_polished": polished["overview"],
        "why_best_overall_polished": polished["why_best_overall"]
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(final_output, f, indent=2)

    print("\n--- POLISHED OVERVIEW ---\n")
    print(final_output["overview_polished"])
    print("\n--- POLISHED WHY BEST OVERALL ---\n")
    print(final_output["why_best_overall_polished"])
    print(f"\nSaved {OUTPUT_JSON}")


if __name__ == "__main__":
    main()

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- POLISHED OVERVIEW ---

For 'wireless headphones for studying', the strongest results emphasize wireless, microphone, over-ear, portable.

The top-ranked products combine query relevance with personalized ranking signals, while buyer feedback helps distinguish strong all-round options from more niche alternatives.

The leading option is reasonably rated, with an average rating of 4.0 from 74 reviews.

Quick picks:
- Best overall: Wireless Kids Headphones with Microphones
- Best value: Kids Wireless Headphones
- Best For School: SHON Kids Wireless Fold On
- Best For Home Use: Comfortable Portable Wireless Headphones Hi
- Best For Long Sessions: EAORUL 100H Playtime Wireless Headphones Over...

--- POLISHED WHY BEST OVERALL ---

Wireless Kids Headphones with Microphones ranks first because it combines wireless, microphone, over-ear with the strongest overall blend of relevance, personalized ranking, and buyer confidence signals. It is rated 4.0 from 74 reviews. A large share of its r